# Global high-rise building fire map (2000–2026)

本 Notebook 从清洗后的事件 CSV 出发，默认可在无 GEE 凭据时生成火灾发生地图；设置服务账号后，可调用 Earth Engine 为每个事件点补充 2020 年人口密度与事件日前 7 天热浪指标，再以点大小编码人口密度、点颜色编码热浪程度。

**数据边界**：部分坐标是城市代表点而非精确火场；公开资料在亚洲、欧洲和重大事故中更密集，因此本图展示的是“已记录事件分布”，不是各地区真实发生率。

## 1. 首次安装依赖

只需运行一次。若服务器环境已预装这些包，可以跳过。

In [ ]:
import subprocess,sys # 导入子进程与当前解释器路径
INSTALL_DEPS=False # 首次运行时改为 True，安装成功后再改回 False
packages=['pandas','numpy','matplotlib','cartopy','earthengine-api','google-auth'] # 声明本项目所需的最小依赖
if INSTALL_DEPS: subprocess.check_call([sys.executable,'-m','pip','install','--upgrade',*packages]) # 仅在显式开启时安装或升级依赖

## 2. 导入库并设置文件路径

每行 Python 代码均带中文注释，代码单元内不保留无意义空行。

In [ ]:
from pathlib import Path # 处理跨平台文件路径
import os # 读取服务账号环境变量
import math # 进行符号大小缩放
import numpy as np # 处理数值数组
import pandas as pd # 读取与整理事件表
import matplotlib as mpl # 设置出版级绘图参数
import matplotlib.pyplot as plt # 绘制静态地图
from matplotlib.lines import Line2D # 构造图例符号
import cartopy.crs as ccrs # 提供全球地图投影
import cartopy.feature as cfeature # 提供 Natural Earth 基础地理要素
import ee # 调用 Google Earth Engine Python API
import requests # 下载 Earth Engine 生成的全球人口栅格预览
from matplotlib.collections import LineCollection # 绘制随热浪等级渐变的拟合趋势线
ROOT=Path.cwd() # 将当前 Notebook 所在目录作为工作目录
CSV_PATH=ROOT/'global_fire_events_2000_2026.csv' # 指定原始清洗 CSV
ENRICHED_CSV_PATH=ROOT/'global_fire_events_2000_2026_gee_enriched.csv' # 指定 GEE 增强 CSV
FIGURE_STEM=ROOT/'global_fire_occurrence_map' # 指定地图输出文件名前缀
POP_BACKGROUND_PATH=ROOT/'gpw2020_population_count_global.png' # 指定全球约一公里人口数量底图路径
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans','sans-serif'],'svg.fonttype':'none','pdf.fonttype':42,'font.size':7,'legend.frameon':False}) # 应用 Nature 风格字体与可编辑矢量文本设置

## 3. 读取并核验事件数据

In [ ]:
df=pd.read_csv(CSV_PATH,encoding='utf-8-sig') # 读取带 BOM 的 UTF-8 CSV
df['event_date']=pd.to_datetime(df['event_date'],errors='coerce') # 将事件日期转换为时间类型
df['latitude']=pd.to_numeric(df['latitude'],errors='coerce') # 将纬度转换为数值
df['longitude']=pd.to_numeric(df['longitude'],errors='coerce') # 将经度转换为数值
required={'event_id','event_date','latitude','longitude','status_group'} # 定义制图所需字段
missing=required.difference(df.columns) # 检查字段是否缺失
assert not missing,f'CSV 缺少字段: {sorted(missing)}' # 在字段不完整时立即停止
valid_geo=df['latitude'].between(-90,90)&df['longitude'].between(-180,180) # 标记合法经纬度
print(f'总事件数: {len(df):,}; 可制图事件数: {int(valid_geo.sum()):,}; 日期范围: {df.event_date.min().date()} 至 {df.event_date.max().date()}') # 输出紧凑的数据核验摘要

## 4. 配置 Service Account / Google Cloud Project

1. 在 Google Cloud 项目中启用 Earth Engine API，并完成 Earth Engine 项目注册。
2. 为服务账号创建 JSON 密钥，勿提交到 Git。生产服务器优先使用 Application Default Credentials。
3. 设置环境变量：GEE_PROJECT_ID、GEE_SERVICE_ACCOUNT、GOOGLE_APPLICATION_CREDENTIALS。
4. 将 RUN_GEE 设为 1 后运行以下单元。

In [ ]:
GEE_PROJECT_ID=os.getenv('GEE_PROJECT_ID','your-google-cloud-project-id') # 读取 Google Cloud 项目 ID
GEE_SERVICE_ACCOUNT=os.getenv('GEE_SERVICE_ACCOUNT','your-service-account@your-project.iam.gserviceaccount.com') # 读取服务账号邮箱
GEE_KEY_FILE=Path(os.getenv('GOOGLE_APPLICATION_CREDENTIALS','service-account-key.json')) # 读取服务账号 JSON 密钥路径
RUN_GEE=os.getenv('RUN_GEE','0')=='1' # 使用环境变量显式控制是否运行 GEE
WINDOW_DAYS=7 # 定义事件日前含事件日的热浪窗口天数
ABS_HEAT_THRESHOLD_C=30.0 # 设置全球统一的最低高温阈值以避免寒冷地区低百分位被误判
GEE_MAX_EVENTS=int(os.getenv('GEE_MAX_EVENTS','0')) # 默认处理全部事件，可设置正整数限制事件数进行快速测试
GEE_WORKERS=int(os.getenv('GEE_WORKERS','2')) # 设置并发事件数，默认两个线程以兼顾速度和配额稳定性
GEE_RETRIES=int(os.getenv('GEE_RETRIES','3')) # 设置单事件失败后的最大重试次数
USE_EXISTING_ENRICHED=os.getenv('USE_EXISTING_ENRICHED','0')=='1' # 允许跳过 GEE 并直接使用已生成的增强 CSV 重绘地图
def initialize_ee(): # 定义无人值守认证函数
    if GEE_PROJECT_ID.startswith('your-'): raise ValueError('请先设置 GEE_PROJECT_ID') # 阻止使用占位项目 ID
    if GEE_KEY_FILE.exists() and not GEE_SERVICE_ACCOUNT.startswith('your-'): # 判断是否使用本地 JSON 密钥
        credentials=ee.ServiceAccountCredentials(GEE_SERVICE_ACCOUNT,str(GEE_KEY_FILE)) # 构造服务账号凭据
        ee.Initialize(credentials=credentials,project=GEE_PROJECT_ID) # 使用服务账号与 Cloud Project 初始化 Earth Engine
    else: # 在云主机或已配置 ADC 的电脑上走默认凭据
        import google.auth # 延迟导入 Google 身份认证库
        credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/earthengine']) # 获取 Application Default Credentials
        ee.Initialize(credentials=credentials,project=GEE_PROJECT_ID) # 使用默认凭据初始化 Earth Engine
    return ee.String('Earth Engine connection OK').getInfo() # 发起轻量请求验证连接
print(initialize_ee() if RUN_GEE else 'RUN_GEE=0：跳过 GEE，仍可生成未增强的全球发生地图') # 根据开关执行或跳过认证

## 5. 通过 GEE 计算人口密度与热浪指标

人口数据同时读取原脚本中的 GPW 2020 population count 和官方 population density 产品。热浪指标使用 ERA5-Land 日最高 2 米温度：事件日前 7 天中，高于事件月份 1991–2020 本地第 90 百分位且高于 30 °C 的天数与累计超阈值温度。该指标仅代表时空关联，不代表火灾因果关系。

In [ ]:
from concurrent.futures import ThreadPoolExecutor,as_completed # 导入线程池以并发执行独立事件请求
import time # 导入等待函数以支持失败重试
def build_gee_sources(): # 定义 GEE 数据源构建函数
    pop_count=ee.ImageCollection('CIESIN/GPWv411/GPW_Population_Count').filterDate('2020-01-01','2021-01-01').first().select('population_count') # 读取 GPW 2020 每像元人口数
    pop_density=ee.ImageCollection('CIESIN/GPWv411/GPW_Population_Density').filterDate('2020-01-01','2021-01-01').first().select('population_density') # 读取 GPW 2020 人口密度
    era5=ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').select('temperature_2m_max') # 读取 ERA5-Land 日最高 2 米温度
    return pop_count,pop_density,era5 # 返回三个服务端对象
def sample_image_at_region(image,region): # 定义单幅 ERA5-Land 影像的局地采样
    value=image.reduceRegion(reducer=ee.Reducer.mean(),geometry=region,scale=11132,bestEffort=True,maxPixels=100000,tileScale=4).get('temperature_2m_max') # 在事件邻域提取日最高温
    return image.set('local_tmax_c',ee.Number(value).subtract(273.15)) # 将摄氏温度写入影像属性
def process_event(row,pop_count,pop_density,era5): # 定义单事件的低内存 GEE 计算
    point=ee.Geometry.Point([float(row.longitude),float(row.latitude)]) # 创建事件点几何
    region=point.buffer(5500) # 使用约半个 ERA5-Land 像元的邻域采样气温
    event_date=ee.Date(row.event_date.strftime('%Y-%m-%d')) # 将本地事件日期转换为 GEE 日期
    event_month=int(row.event_date.month) # 读取事件月份用于气候基线筛选
    climatology=era5.filterDate('1991-01-01','2021-01-01').filter(ee.Filter.calendarRange(event_month,event_month,'month')) # 选择 1991–2020 同月日最高温
    climate_values=ee.List(climatology.map(lambda image:sample_image_at_region(image,region)).aggregate_array('local_tmax_c')) # 先局地采样再聚合一维温度序列
    p90_c=ee.Number(climate_values.reduce(ee.Reducer.percentile([90]))) # 计算局地同月第 90 百分位
    threshold_c=p90_c.max(ABS_HEAT_THRESHOLD_C) # 取相对阈值和 30 摄氏度阈值中的较高值
    window_start=event_date.advance(-(WINDOW_DAYS-1),'day') # 计算事件窗口起点
    window_end=event_date.advance(1,'day') # 将结束日期推进一天以包含事件日
    event_values=ee.List(era5.filterDate(window_start,window_end).map(lambda image:sample_image_at_region(image,region)).aggregate_array('local_tmax_c')) # 获取事件七天局地温度序列
    heat_days=ee.Number(event_values.map(lambda value:ee.Number(value).gt(threshold_c)).reduce(ee.Reducer.sum())) # 统计超过复合阈值的日数
    degree_days=ee.Number(event_values.map(lambda value:ee.Number(value).subtract(threshold_c).max(0)).reduce(ee.Reducer.sum())) # 累计超过阈值的温度
    max_temp=ee.Number(event_values.reduce(ee.Reducer.max())) # 计算七天窗口最高温
    population=ee.Image.cat([pop_count,pop_density]).reduceRegion(reducer=ee.Reducer.mean(),geometry=point.buffer(1000),scale=1000,bestEffort=True,maxPixels=100000,tileScale=4) # 提取约一公里邻域人口指标
    result=ee.Dictionary({'event_id':str(row.event_id),'population_count_2020':population.get('population_count'),'population_density_2020_per_km2':population.get('population_density'),'heatwave_days_7d':heat_days,'heatwave_max_t2m_c':max_temp,'heatwave_p90_t2m_c':p90_c,'heatwave_degree_days_c':degree_days,'heatwave_score_0_100':degree_days.multiply(10).min(100).max(0),'gee_enrichment_status':'completed'}) # 汇总事件指标
    return result.getInfo() # 一次请求取回当前事件全部指标
def process_event_with_retry(row,pop_count,pop_density,era5): # 定义带重试的事件处理包装器
    for attempt in range(1,GEE_RETRIES+1): # 按配置次数尝试当前事件
        try: return process_event(row,pop_count,pop_density,era5) # 成功时立即返回指标
        except Exception as error: # 捕获单事件的服务端或网络异常
            if attempt==GEE_RETRIES: return {'event_id':str(row.event_id),'gee_enrichment_status':f'failed: {type(error).__name__}'} # 重试耗尽后保留失败状态
            time.sleep(attempt*5) # 使用递增等待时间降低瞬时配额压力
def download_population_background(pop_count): # 定义全球约一公里人口数量底图下载函数
    log_population=pop_count.updateMask(pop_count.gt(0)).log10() # 对正人口像元取常用对数以显示全球数量级差异
    visualized=log_population.visualize(min=0,max=5,palette=['F3F1EC','D4E1E5','9FC4CF','6696AA','345E78','163A51']) # 应用低饱和蓝灰人口色带
    region=ee.Geometry.Rectangle([-180,-90,180,90],geodesic=False) # 定义完整全球输出范围
    url=visualized.getThumbURL({'region':region,'dimensions':'4096x2048','format':'png','crs':'EPSG:4326'}) # 请求全球显示分辨率的人口栅格
    response=requests.get(url,timeout=300) # 下载 GEE 返回的人口栅格预览
    response.raise_for_status() # 在网络或服务端错误时立即停止
    POP_BACKGROUND_PATH.write_bytes(response.content) # 保存带透明掩膜的人口底图
def run_gee_enrichment(frame): # 定义并发低内存 GEE 增强流程
    pop_count,pop_density,era5=build_gee_sources() # 初始化人口与气象数据源
    if not POP_BACKGROUND_PATH.exists(): download_population_background(pop_count) # 首次运行时下载 GPW 2020 全球人口数量底图
    latest=pd.to_datetime(ee.Date(era5.aggregate_max('system:time_start')).format('YYYY-MM-dd').getInfo()) # 查询 ERA5-Land 当前最新日期
    work=frame.loc[valid_geo&frame['event_date'].notna()&frame['event_date'].le(latest)].copy() # 保留坐标有效且有气象覆盖的事件
    if GEE_MAX_EVENTS>0: work=work.head(GEE_MAX_EVENTS).copy() # 允许限制事件数进行端到端快速测试
    rows=list(work.itertuples(index=False)) # 将待处理记录转换为可提交任务的列表
    records=[] # 创建指标结果容器
    with ThreadPoolExecutor(max_workers=GEE_WORKERS) as executor: # 建立受控并发线程池
        futures={executor.submit(process_event_with_retry,row,pop_count,pop_density,era5):str(row.event_id) for row in rows} # 为每个事件提交独立请求
        for completed,future in enumerate(as_completed(futures),1): # 按请求完成顺序收集结果
            record=future.result() # 读取当前事件返回的指标字典
            records.append(record) # 保存当前事件指标
            print(f'GEE 已完成 {completed}/{len(rows)}：{record["event_id"]}，状态 {record["gee_enrichment_status"]}') # 输出事件级进度与状态
    metrics=pd.DataFrame(records).drop_duplicates('event_id') # 整理并去重 GEE 指标
    base=frame.drop(columns=[column for column in metrics.columns if column!='event_id' and column in frame.columns]) # 删除将由真实结果替换的占位列
    result=base.merge(metrics,on='event_id',how='left') # 按事件 ID 合并指标
    result['gee_enrichment_status']=result['gee_enrichment_status'].fillna('not_processed') # 标记因测试限制等原因未处理的记录
    valid_result=result['latitude'].between(-90,90)&result['longitude'].between(-180,180)&result['event_date'].notna() # 重新检查合并后坐标和日期
    result.loc[~valid_result,'gee_enrichment_status']='invalid_coordinate_or_date' # 标记无法空间计算的记录
    result.loc[result['event_date'].gt(latest),'gee_enrichment_status']='era5_not_yet_available' # 标记超出 ERA5-Land 当前覆盖的记录
    result.to_csv(ENRICHED_CSV_PATH,index=False,encoding='utf-8-sig') # 写出 GEE 增强 CSV
    return result # 返回增强后的 DataFrame
if RUN_GEE: df=run_gee_enrichment(df) # 在启用 GEE 时重新计算并写出增强结果
elif USE_EXISTING_ENRICHED and ENRICHED_CSV_PATH.exists(): df=pd.read_csv(ENRICHED_CSV_PATH,encoding='utf-8-sig') # 在重绘时直接读取已有增强 CSV
if 'event_date' in df.columns: df['event_date']=pd.to_datetime(df['event_date'],errors='coerce') # 确保增强 CSV 的事件日期恢复为时间类型
print(f'当前制图数据: {ENRICHED_CSV_PATH.name if (RUN_GEE or USE_EXISTING_ENRICHED) else CSV_PATH.name}') # 显示地图使用的数据文件


## 6. 绘制 Nature Cities 风格全球发生地图

若已完成 GEE 增强：全球底图颜色表示 GPW 2020 每个原生 30 角秒（约 1×1 km）像元的人口数量，火灾点颜色表示事件日前 7 天热浪评分。全球期刊图无法逐一分辨数亿个 1 km 像元，因此显示层由 GEE 按画布尺度金字塔化，但计算与数据口径仍保留原生约 1 km 网格。

In [ ]:
plot_df=df.loc[df['latitude'].between(-90,90)&df['longitude'].between(-180,180)].copy() # 筛选可制图的事件点
metric_mode={'population_count_2020','heatwave_score_0_100'}.issubset(plot_df.columns) and plot_df['population_count_2020'].notna().any() and plot_df['heatwave_score_0_100'].notna().any() # 判断是否具备 GEE 指标
population_mode=metric_mode and POP_BACKGROUND_PATH.exists() # 判断约一公里人口数量底图是否存在
fig=plt.figure(figsize=(7.2,4.35),facecolor='white') # 创建 Nature 双栏宽度附近的画布
ax=fig.add_axes([0.035,0.22,0.93,0.69],projection=ccrs.Robinson()) # 为地图和双指标色条预留空间
ax.set_global() # 显示全球范围
ax.spines['geo'].set_edgecolor('#AAA8A4') # 将地图外框调整为浅灰色
ax.spines['geo'].set_linewidth(0.45) # 将地图外框调整为期刊式细线
ax.add_feature(cfeature.LAND.with_scale('110m'),facecolor='#F1F0ED',edgecolor='none',zorder=0) # 绘制克制的浅灰陆地背景
ax.add_feature(cfeature.OCEAN.with_scale('110m'),facecolor='#FFFFFF',edgecolor='none',zorder=0) # 使用白色海洋保持期刊版面感
if population_mode: # 在 GEE 指标和人口底图均存在时绘制双指标地图
    population_image=plt.imread(POP_BACKGROUND_PATH) # 读取 GEE 生成的全球人口数量栅格
    ax.imshow(population_image,origin='upper',extent=[-180,180,-90,90],transform=ccrs.PlateCarree(),interpolation='bilinear',zorder=0.5) # 将原生约一公里人口数据的显示层投影到 Robinson 地图
ax.add_feature(cfeature.COASTLINE.with_scale('110m'),edgecolor='#8F8D89',linewidth=0.35,zorder=1) # 在人口栅格上方绘制海岸线
ax.add_feature(cfeature.BORDERS.with_scale('110m'),edgecolor='#C4C2BE',linewidth=0.18,zorder=1) # 绘制弱化国界
if population_mode: # 使用火灾点颜色编码热浪评分
    heat=pd.to_numeric(plot_df['heatwave_score_0_100'],errors='coerce').fillna(0).clip(0,100) # 清洗热浪评分
    heat_cmap=mpl.colors.LinearSegmentedColormap.from_list('nature_heat',['#F6C87A','#F09A62','#DF614F','#B23B4C','#702B45']) # 使用与蓝灰人口底图区分的暖色带
    scatter=ax.scatter(plot_df['longitude'],plot_df['latitude'],s=9+27*np.power(heat.to_numpy(dtype=float)/100,0.65),c=heat,cmap=heat_cmap,vmin=0,vmax=100,alpha=0.45+0.50*np.power(heat.to_numpy(dtype=float)/100,0.65),edgecolors='white',linewidths=0.35,transform=ccrs.PlateCarree(),zorder=3) # 使主地图热浪分数越高的火灾气泡越深越不透明且面积越大
    top_heat_events=plot_df.assign(_heat_label=pd.to_numeric(plot_df['heatwave_score_0_100'],errors='coerce')).dropna(subset=['_heat_label']).nlargest(5,'_heat_label') # 选择热浪评分最高的五条城市火灾记录
    label_offsets={'Melbourne':(-38,-13),'Luoyang':(8,22),'Dubai':(-34,-18),'Nanjing':(24,-4),'Islamabad':(-28,15)} # 为香港标签腾出空间并保持五个最高热浪城市标签不交叉
    for _,event in top_heat_events.iterrows(): # 逐条绘制五个最高热浪评分城市标签
        city_name=str(event['location']).split(',')[0].strip() or str(event['country']) # 从地点字段提取简洁英文城市名
        label_offset=label_offsets.get(city_name,(8,8)) # 获取当前城市的标签偏移量
        ax.annotate(city_name,xy=(float(event['longitude']),float(event['latitude'])),xycoords=ccrs.PlateCarree()._as_mpl_transform(ax),xytext=label_offset,textcoords='offset points',ha='left' if label_offset[0]>=0 else 'right',va='center',fontsize=4.4,fontweight='bold',color='#272727',arrowprops={'arrowstyle':'-','color':'#8F8D89','linewidth':0.4,'shrinkA':1.5,'shrinkB':2.5},zorder=5) # 仅以统一黑色字体和极细引导线标注英文城市名
    hong_kong_events=plot_df.loc[plot_df['location'].fillna('').astype(str).str.contains('Hong Kong',case=False,regex=False)].copy() # 筛选具有有效坐标的香港火灾记录
    if not hong_kong_events.empty: # 仅在数据中确实存在香港有效点时添加标签
        hong_kong_lon=float(pd.to_numeric(hong_kong_events['longitude'],errors='coerce').median()) # 使用有效香港记录的经度中位数确定标签锚点
        hong_kong_lat=float(pd.to_numeric(hong_kong_events['latitude'],errors='coerce').median()) # 使用有效香港记录的纬度中位数确定标签锚点
        ax.annotate('Hong Kong',xy=(hong_kong_lon,hong_kong_lat),xycoords=ccrs.PlateCarree()._as_mpl_transform(ax),xytext=(24,-17),textcoords='offset points',ha='left',va='center',fontsize=4.4,fontweight='bold',color='#272727',arrowprops={'arrowstyle':'-','color':'#8F8D89','linewidth':0.4,'shrinkA':1.5,'shrinkB':2.5},zorder=5) # 用黑色英文城市名标注香港有效点簇
    population_cmap=mpl.colors.LinearSegmentedColormap.from_list('population_count',['#F3F1EC','#D4E1E5','#9FC4CF','#6696AA','#345E78','#163A51']) # 复现人口数量底图色带
    population_scalar=mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0,vmax=5),cmap=population_cmap) # 建立人口数量对数色标对象
    population_axis=fig.add_axes([0.10,0.115,0.34,0.022]) # 在左下方建立人口数量色条轴
    population_bar=fig.colorbar(population_scalar,cax=population_axis,orientation='horizontal') # 绘制人口数量颜色图例
    population_bar.set_ticks([0,1,2,3,4,5]) # 设置人口数量的对数刻度
    population_bar.set_ticklabels(['1','10','100','1k','10k','100k']) # 将对数刻度显示为每像元人数
    population_bar.set_label('Population count per ~1-km cell (GPW 2020)',fontsize=6.5,labelpad=3) # 标注约一公里像元人口数量
    population_bar.ax.tick_params(labelsize=6,length=2,width=0.5) # 设置人口色条刻度样式
    heat_axis=fig.add_axes([0.56,0.115,0.34,0.022]) # 在右下方建立热浪评分色条轴
    heat_bar=fig.colorbar(scatter,cax=heat_axis,orientation='horizontal') # 绘制热浪评分颜色图例
    heat_bar.set_label('Fire-event heatwave score (0–100)',fontsize=6.5,labelpad=3) # 标注火灾事件热浪评分
    heat_bar.ax.tick_params(labelsize=6,length=2,width=0.5) # 设置热浪色条刻度样式
else: # 在尚未生成人口底图时采用核验状态编码
    groups=[('严格确认','Strictly verified','#B64342',18,0.88),('确认但有待复核字段','Verified; fields pending review','#D98B70',15,0.82),('专题汇编记录','Compendium record','#E6B9A8',11,0.64),('候选待核实','Candidate','#767676',10,0.58),('边界/建议排除','Boundary / exclude','#B8B8B8',9,0.50)] # 定义状态原值、英文图例与视觉层级
    handles=[] # 创建图例句柄容器
    for status,label,color,size,alpha in groups: # 逐类绘制事件点
        subset=plot_df.loc[plot_df['status_group'].eq(status)] # 按原始中文状态值筛选记录
        if subset.empty: continue # 跳过当前数据中不存在的类别
        ax.scatter(subset['longitude'],subset['latitude'],s=size,c=color,alpha=alpha,edgecolors='white',linewidths=0.25,transform=ccrs.PlateCarree(),zorder=3) # 绘制当前核验状态的事件点
        handles.append(Line2D([0],[0],marker='o',linestyle='none',markerfacecolor=color,markeredgecolor='white',markeredgewidth=0.35,markersize=math.sqrt(size),label=f'{label} (n={len(subset)})')) # 生成与地图一致的英文图例符号
    fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(0.5,0.075),ncol=3,fontsize=6.2,handletextpad=0.35,columnspacing=1.0) # 将共享图例置于地图下方
if metric_mode: # 在存在热浪指标时绘制左下角描述性关联子图
    association=plot_df[['heatwave_score_0_100']].copy() # 提取热浪评分用于等级频次与气泡编码
    association['heatwave_score_0_100']=pd.to_numeric(association['heatwave_score_0_100'],errors='coerce').fillna(0).clip(0,100) # 清洗热浪评分
    level_labels=['0','1–25','26–50','51–75','76–100'] # 定义热浪等级显示标签
    association['heat_level']=pd.cut(association['heatwave_score_0_100'],bins=[-0.001,0.001,25,50,75,100.001],labels=level_labels,include_lowest=True) # 将热浪评分分为五个等级
    summary=association.groupby('heat_level',observed=False).agg(frequency=('heatwave_score_0_100','size')).reindex(level_labels).fillna(0) # 汇总五个热浪等级的火灾事件频次
    inset_x=np.array([0,25,50,75,100],dtype=float) # 使用五个热浪分数刻度作为真实横轴坐标
    inset_sizes=14+44*np.power(inset_x/100,0.75) # 使热浪分数越高的观测气泡面积越大
    inset_frequency=summary['frequency'].to_numpy(dtype=float) # 提取五个热浪等级的观测火灾频次
    inset_cmap=mpl.colors.LinearSegmentedColormap.from_list('inset_heat',['#F6C87A','#F09A62','#DF614F','#B23B4C','#702B45']) # 复用主图暖色体系建立气泡与拟合线渐变
    fit_x=np.linspace(0,100,240) # 创建覆盖完整热浪评分范围的平滑拟合横轴
    fit_total_variance=float(np.sum((inset_frequency-inset_frequency.mean())**2)) # 计算原始频次的总离差平方和
    fit_candidates={} # 创建候选拟合函数及其原始尺度决定系数容器
    for fit_degree,fit_name in [(1,'Linear'),(2,'Quadratic'),(3,'Cubic')]: # 比较一至三次多项式并避免五点四次强制插值
        polynomial_coefficients=np.polyfit(inset_x,inset_frequency,fit_degree) # 拟合当前阶数的多项式系数
        polynomial_observed=np.polyval(polynomial_coefficients,inset_x) # 计算当前多项式在观测点的预测值
        polynomial_smooth=np.polyval(polynomial_coefficients,fit_x) # 计算当前多项式的平滑趋势曲线
        polynomial_r2=1-float(np.sum((inset_frequency-polynomial_observed)**2))/fit_total_variance # 在原始火灾频次尺度计算决定系数
        fit_candidates[fit_name]=(polynomial_r2,polynomial_smooth) # 保存当前多项式的决定系数和曲线
    exponential_coefficients=np.polyfit(inset_x,np.log1p(inset_frequency),1) # 在对数频次空间拟合指数候选函数
    exponential_observed=np.expm1(np.polyval(exponential_coefficients,inset_x)) # 计算指数候选函数的观测点预测值
    exponential_smooth=np.expm1(np.polyval(exponential_coefficients,fit_x)) # 计算指数候选函数的平滑曲线
    exponential_r2=1-float(np.sum((inset_frequency-exponential_observed)**2))/fit_total_variance # 计算指数候选函数的原始尺度决定系数
    fit_candidates['Exponential']=(exponential_r2,exponential_smooth) # 保存指数候选函数的决定系数和曲线
    logarithmic_coefficients=np.polyfit(np.log1p(inset_x),inset_frequency,1) # 使用对数热浪评分拟合对数候选函数
    logarithmic_observed=np.polyval(logarithmic_coefficients,np.log1p(inset_x)) # 计算对数候选函数的观测点预测值
    logarithmic_smooth=np.polyval(logarithmic_coefficients,np.log1p(fit_x)) # 计算对数候选函数的平滑曲线
    logarithmic_r2=1-float(np.sum((inset_frequency-logarithmic_observed)**2))/fit_total_variance # 计算对数候选函数的原始尺度决定系数
    fit_candidates['Logarithmic']=(logarithmic_r2,logarithmic_smooth) # 保存对数候选函数的决定系数和曲线
    best_fit_name,(best_fit_r2,fit_y)=max(fit_candidates.items(),key=lambda item:item[1][0]) # 按最高决定系数自动选择最佳候选函数
    fit_y=np.clip(fit_y,0,inset_frequency.max()*1.05) # 限制拟合曲线为非负并抑制边界过冲
    fit_points=np.column_stack([fit_x,fit_y]).reshape(-1,1,2) # 将平滑拟合点转换为渐变线段端点
    fit_segments=np.concatenate([fit_points[:-1],fit_points[1:]],axis=1) # 构建连续拟合趋势线段
    inset=fig.add_axes([0.078,0.275,0.205,0.22],facecolor=(0.985,0.985,0.98,0.97),zorder=5) # 缩短并略微加高左下角关联子图
    inset.plot(inset_x,inset_frequency,color='#C4C2BE',linewidth=0.6,linestyle=(0,(2,2)),zorder=1) # 使用浅灰虚线连接五个观测频次
    gradient_fit=LineCollection(fit_segments,cmap=inset_cmap,norm=mpl.colors.Normalize(vmin=0,vmax=100),linewidths=1.45,zorder=2) # 建立覆盖零至一百热浪分数的渐变拟合线
    gradient_fit.set_array(fit_x[:-1]) # 将拟合横坐标映射到热浪暖色渐变
    inset.add_collection(gradient_fit) # 将最佳平滑拟合趋势叠加到观测散点下方
    inset.scatter(inset_x,inset_frequency,s=inset_sizes,c=inset_x,cmap=inset_cmap,norm=mpl.colors.Normalize(vmin=0,vmax=100),alpha=0.42+0.53*np.power(inset_x/100,0.75),edgecolors='white',linewidths=0.45,zorder=3) # 使热浪分数越高的气泡颜色越深透明度越低且面积越大
    inset.set_xticks([0,25,50,75,100],['0','25','50','75','100'],fontsize=4.5) # 仅显示五个指定热浪分数刻度
    inset.set_xlabel('Heatwave score',fontsize=4.8,labelpad=1.5,color='#272727') # 标注归因图横轴变量
    inset.set_ylabel('Fire events',fontsize=5) # 标注火灾事件频次轴
    inset.set_title('Attribution plot',loc='left',fontsize=5.5,fontweight='bold',pad=3,color='#272727') # 使用归因图的英文标题
    inset.tick_params(axis='y',labelsize=4.5,length=2,width=0.4) # 设置纵轴刻度样式
    inset.tick_params(axis='x',length=0) # 隐藏横轴刻度线以节省空间
    inset.spines[['top','right','bottom','left']].set_visible(True) # 显示归因图完整外框
    inset.spines[['top','right','bottom','left']].set_color('#AAA8A4') # 使用与主地图外框一致的浅灰色
    inset.spines[['top','right','bottom','left']].set_linewidth(0.45) # 统一主图和归因图的外框线宽
    inset.set_xlim(-3,105) # 覆盖零至一百热浪分数并为两端气泡保留边距
    inset.set_ylim(-4,max(float(summary['frequency'].max())*1.16,10)) # 为观测气泡与拟合曲线保留稳定的顶部留白
    inset.text(0.97,0.92,f'{best_fit_name}; R²={best_fit_r2:.3f}',transform=inset.transAxes,ha='right',va='top',fontsize=3.8,color='#6B6966') # 标明最高决定系数对应的拟合函数与数值
    inset.text(0.0,-0.39,'Bubble size and opacity increase with heatwave score',transform=inset.transAxes,fontsize=3.9,color='#6B6966') # 解释热浪分数的气泡面积与透明度编码
ax.set_title('Global distribution of documented high-rise building fires, 2000–2026',loc='center',fontsize=10,fontweight='bold',pad=7,color='#272727') # 居中显示主图标题
fig.text(0.035,0.012,'Population source: native 30-arcsec (~1-km) GPW 2020 cells; global display is pyramided.',ha='left',va='bottom',fontsize=6,color='#6B6966') # 添加人口网格解释
fig.savefig(FIGURE_STEM.with_suffix('.svg'),bbox_inches='tight') # 首先保存文本可编辑的 SVG
fig.savefig(FIGURE_STEM.with_suffix('.pdf'),bbox_inches='tight') # 保存可投稿的矢量 PDF
fig.savefig(FIGURE_STEM.with_suffix('.png'),dpi=400,bbox_inches='tight',facecolor='white') # 保存高分辨率预览 PNG
fig.savefig(FIGURE_STEM.with_suffix('.tiff'),dpi=600,bbox_inches='tight',facecolor='white') # 保存 600 dpi TIFF
plt.show() # 在 Notebook 中显示最终地图

## 7. 输出检查

确认图中文字在 100% 缩放下可读、点未越界、SVG/PDF 文字可选择；若运行了 GEE，还应抽查人口密度与热浪字段的缺失率及极值。

In [ ]:
print(plot_df[['latitude','longitude']].describe().round(2)) # 检查坐标范围与异常值
print(plot_df['status_group'].value_counts(dropna=False)) # 检查各核验状态样本量
if metric_mode: print(plot_df[['population_density_2020_per_km2','heatwave_score_0_100']].describe().round(2)) # 在 GEE 模式下检查两项制图指标
print([str(FIGURE_STEM.with_suffix(ext)) for ext in ['.svg','.pdf','.png','.tiff']]) # 列出最终地图输出路径